java -jar PerformanceMatcher.jar -b -q -D --use-chroma-map -os alignment_path.txt /Users/server/Desktop/match/wav/bach5_mov1_S_tsm.wav /Users/server/Desktop/match/wav/bach5_mov1_S.wav >> dump_features_testagain.txt

In [1]:
import subprocess
import tempfile
import os
import pickle
import numpy as np
import re

In [2]:
import subprocess
import re
from pathlib import Path
from typing import List, Tuple, Sequence, Optional

import librosa as lb
import matplotlib.pyplot as plt


def parse_alignment(stdout_text: str) -> List[Tuple[int, int]]:
    """Extract (query_frame, ref_frame) tuples from stdout text."""
    pattern = re.compile(r"^ALIGNMENT\s+(\d+)\s*:?[\s,]+(\d+)\s*$")
    pairs = []
    for line in stdout_text.splitlines():
        line = line.strip()
        m = pattern.match(line)
        if m:
            pairs.append((int(m.group(1)), int(m.group(2))))
    return pairs


def run_match(
    jar_path: str,
    wav1_path: str,  # query wav (time-stretched)
    wav2_path: str,  # reference wav
    dump_output_to: str = "dump_features_testagain.txt",
    graph: bool = False,
    sr: int = 22050,
    hop_length: int = 441,
    shade_silence_secs: Sequence[Tuple[float, Optional[float]]] = ((0.0, 3.9), (195.4, None)),
    marker_size: float = 0.2,
) -> List[Tuple[int, int]]:
    """
    Run the matcher .jar and return alignment as a list of (query_frame, ref_frame).
    Equivalent to:
      java -jar match/PerformanceMatcher.jar -b -q -D --use-chroma-map -os [wav1] [wav2]
    """
    # Always-necessary flags
    flags = ["-b", "-q", "-D", "--use-chroma-map", "-os"]

    cmd = [
        "java",
        "-jar",
        str(Path(jar_path)),
        *flags,
        str(Path(wav1_path)),
        str(Path(wav2_path)),
    ]

    # Run and capture stdout/stderr
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    raw_text = proc.stdout or proc.stderr
    Path(dump_output_to).write_text(raw_text, encoding="utf-8")

    # Parse alignment
    # alignment = parse_alignment(raw_text)

    # # Optional visualization
    # if graph:
    #     query_y, _ = lb.load(wav1_path, sr=sr)
    #     ref_y, _ = lb.load(wav2_path, sr=sr)

    #     plt.figure()
    #     if alignment:
    #         q_vals, r_vals = zip(*alignment)
    #         plt.plot(q_vals, r_vals, "ro", markersize=marker_size)

    #     plt.xlabel("Query frame number")
    #     plt.ylabel("Reference frame number")

    #     ref_frames_total = len(ref_y) // hop_length

    #     def sec_to_frames(s): return int(round(s * sr / hop_length))

    #     for (start, end) in shade_silence_secs:
    #         y_min = sec_to_frames(start)
    #         y_max = sec_to_frames(end) if end is not None else ref_frames_total
    #         plt.axhspan(ymin=y_min, ymax=y_max, color="blue", alpha=0.5)

    #     plt.legend(["Alignment path"])
    #     plt.title("Query vs Reference Alignment")
    #     plt.tight_layout()
    #     plt.show()


In [3]:
run_match(
    jar_path="/home/slubis/ttmp/PianoConcertoAccompaniment/match/PerformanceMatcher.jar",
    wav1_path="/home/slubis/ttmp/PianoConcertoAccompaniment/scenarios/random/s5/p.wav",
    wav2_path="/home/slubis/ttmp/PianoConcertoAccompaniment/scenarios/random/s5/pref.wav",
    dump_output_to="/home/slubis/ttmp/PianoConcertoAccompaniment/match/alignment_dump.txt",
    graph=True,
)
